# S6E4 — 02 Fold creation

> Teammate's notebook from the team's Kaggle pipeline, cleaned for the repository: empty cells, `!ls` calls, duplicates and commented-out code removed; the code that ran is unchanged.

| | |
|---|---|
| **Input** | `train.csv` |
| **Output** | `fold_idxs.pkl`: 10 folds × 3 seeds (42, 2026, 0), `ohe.pkl` |

**Why:** every model of the ensemble must use the same folds, so that their out-of-fold predictions can be stacked. Instead of stratifying by the target only, rows are stratified by **32 KMeans clusters** built on a 16-component PCA of the one-hot and numeric features, so each fold covers every region of the feature space.

In [1]:
import time
import pandas as pd
import numpy as np
import joblib

# Importing from Scikit-Learn
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
pd.set_option("display.max_columns", None)

In [2]:
data=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
print(data.shape)
print('Memory usage {:5.2f} Mb'.format(data.memory_usage().sum() / 1024**2))
data.head(3)

(630000, 21)
Memory usage 100.94 Mb


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low


In [3]:
TARGET = 'Irrigation_Need'

NUMS = [
    'Soil_pH','Soil_Moisture','Organic_Carbon','Electrical_Conductivity',
    'Temperature_C','Humidity','Rainfall_mm','Sunlight_Hours',
    'Wind_Speed_kmh','Field_Area_hectare','Previous_Irrigation_mm',
]

CATS = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
    'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 
]

In [4]:
import os
import joblib
from tqdm import tqdm

class OneHotEncoder:
    def __init__(self):
        self.mapping = dict()
        self.prefix = ''
        self.postfix = ''
        self.feature_names = []

    def fit(self, data, columns, prefix = '', postfix = '', filepath = ''):
        self.prefix = prefix
        self.postfix = postfix
        
        for col in tqdm(columns):
            unique_values = data[col].unique()

            if len(unique_values) > 1:
                for value in unique_values[:-1]:
                    self.mapping[col] = unique_values
                    self.feature_names.append(self.prefix+f'{col}_{str(value)}'+self.postfix)

        #Save the object
        if len(filepath)>0:
            self.save(filepath)
    
    def transform(self, data):
        
        new_features = dict()
        for col, unique_values in tqdm(self.mapping.items()):
            if len(unique_values) > 1:
                for value in unique_values[:-1]:
                    new_features[self.prefix+f'{col}_{str(value)}'+self.postfix] = data[col].apply(lambda x: 1 if x == value else 0)
        
        features_df = pd.DataFrame(new_features, index=data.index)
        data = pd.concat([data, features_df], axis=1)
        
        return data
                
    def fit_transform(self, data, columns, prefix = '', postfix = '', filepath = ''):
        self.fit(data, columns, prefix, postfix, filepath)
        return self.transform(data)

    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f"OneHotEncoder saved at: {filepath}")

In [5]:
ohe = OneHotEncoder()
ohe.fit(data, CATS, prefix='ohe_', filepath='ohe.pkl')

100%|██████████| 8/8 [00:00<00:00, 29.91it/s]

OneHotEncoder saved at: ohe.pkl


In [6]:
data = ohe.transform(data)

100%|██████████| 8/8 [00:05<00:00,  1.43it/s]


In [7]:
data.head(2)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,ohe_Soil_Type_Loamy,ohe_Soil_Type_Clay,ohe_Soil_Type_Sandy,ohe_Crop_Type_Sugarcane,ohe_Crop_Type_Wheat,ohe_Crop_Type_Rice,ohe_Crop_Type_Potato,ohe_Crop_Type_Cotton,ohe_Crop_Growth_Stage_Sowing,ohe_Crop_Growth_Stage_Vegetative,ohe_Crop_Growth_Stage_Flowering,ohe_Season_Zaid,ohe_Season_Kharif,ohe_Irrigation_Type_Drip,ohe_Irrigation_Type_Rainfed,ohe_Irrigation_Type_Sprinkler,ohe_Water_Source_Rainwater,ohe_Water_Source_River,ohe_Water_Source_Reservoir,ohe_Mulching_Used_No,ohe_Region_East,ohe_Region_South,ohe_Region_North,ohe_Region_West
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low,1,0,0,1,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,1,1,0,0,0
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low,0,1,0,0,1,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,0,1,0,0


In [8]:
data = data[['id', TARGET] + NUMS + ohe.feature_names]
data.head(3)

,id,Irrigation_Need,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,Previous_Irrigation_mm,ohe_Soil_Type_Loamy,ohe_Soil_Type_Clay,ohe_Soil_Type_Sandy,ohe_Crop_Type_Sugarcane,ohe_Crop_Type_Wheat,ohe_Crop_Type_Rice,ohe_Crop_Type_Potato,ohe_Crop_Type_Cotton,ohe_Crop_Growth_Stage_Sowing,ohe_Crop_Growth_Stage_Vegetative,ohe_Crop_Growth_Stage_Flowering,ohe_Season_Zaid,ohe_Season_Kharif,ohe_Irrigation_Type_Drip,ohe_Irrigation_Type_Rainfed,ohe_Irrigation_Type_Sprinkler,ohe_Water_Source_Rainwater,ohe_Water_Source_River,ohe_Water_Source_Reservoir,ohe_Mulching_Used_No,ohe_Region_East,ohe_Region_South,ohe_Region_North,ohe_Region_West
0,0,Low,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,0.82,112.16,1,0,0,1,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,1,1,0,0,0
1,1,Low,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,5.27,47.16,0,1,0,0,1,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,0,1,0,0
2,2,Low,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,8.24,110.38,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,1,0


In [9]:
y_train = data[TARGET]
X_train = data.set_index('id').drop(TARGET, axis='columns')

In [10]:
# Stratifying the data
pca = PCA(n_components=16, random_state=0)
km = KMeans(n_clusters=32, random_state=0)

In [11]:
# First perform PCA to remove correlation
start = time.time()
pca.fit(X_train)
print("Time taken to fit PCA: {} seconds".format(round(time.time() - start, 2)))

Time taken to fit PCA: 0.17 seconds


In [12]:
start = time.time()
X_pca = pca.transform(X_train)
km.fit(X_pca)
print("Time taken to fit km: {} seconds".format(round(time.time() - start, 2)))

Time taken to fit km: 2.97 seconds


In [13]:
y_stratified = km.labels_
print(np.unique(km.labels_, return_counts=True))

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31],
      dtype=int32), array([28369, 30245, 24968, 16679, 20679, 28133,  7831, 18831, 12189,
       13946, 19485, 22496, 20769, 19326, 22768, 29126, 18702, 32430,
       15674,  9354, 20942, 16859, 16916, 21399, 21905, 17334, 16599,
       18598, 17000, 17683, 14278, 18487]))


In [14]:
# Creating your folds for repeated use (for instance, stacking)

folds = 10
seeds = [42, 2026, 0]
fold_idxs = list()

for seed in seeds:
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    fold_idxs.append(list(skf.split(X_train, y_stratified)))

In [15]:
# Checking the produced folds
for j, fold_idxs_ in enumerate(fold_idxs):
    print(f"\n--- seed={seeds[j]} ---")
    for k, (train_idx, validation_idx) in enumerate(fold_idxs_):
        print(f"fold {k} train idxs: {len(train_idx)} validation idxs: {len(validation_idx)} -> {validation_idx[:10]}")


--- seed=42 ---
fold 0 train idxs: 567000 validation idxs: 63000 -> [14 19 20 30 33 45 47 62 71 72]
fold 1 train idxs: 567000 validation idxs: 63000 -> [ 10  12  23  26  46  51  53  61  96 107]
fold 2 train idxs: 567000 validation idxs: 63000 -> [ 13  18  37  48  50  58 117 119 122 143]
fold 3 train idxs: 567000 validation idxs: 63000 -> [  7  24  39  40  43  73  79  97 101 121]
fold 4 train idxs: 567000 validation idxs: 63000 -> [ 1 22 36 49 56 65 66 70 83 90]
fold 5 train idxs: 567000 validation idxs: 63000 -> [ 3  6  8  9 41 63 68 75 80 81]
fold 6 train idxs: 567000 validation idxs: 63000 -> [ 2 27 28 31 38 57 60 76 82 87]
fold 7 train idxs: 567000 validation idxs: 63000 -> [ 0  5 11 15 25 35 52 54 64 69]
fold 8 train idxs: 567000 validation idxs: 63000 -> [16 17 21 32 55 74 77 78 84 93]
fold 9 train idxs: 567000 validation idxs: 63000 -> [  4  29  34  42  44  59  67  91  98 106]

--- seed=2026 ---
fold 0 train idxs: 567000 validation idxs: 63000 -> [ 40  41  46  48  51  52  74  81

In [16]:
joblib.dump(fold_idxs, './fold_idxs.pkl')

['./fold_idxs.pkl']